<a href="https://colab.research.google.com/github/mantrikaran/F1.Stats.Guy/blob/main/Data_Fetch_from_Jolpica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# F1 STATS GUY — DATA DOWNLOAD AGENT (V3)
# Resume-safe. Writes only after full round completion.
# All files stored in: F1 Stats Guy - Jolpica/Jolpica Database
# ═══════════════════════════════════════════════════════════════════════

# ── INSTALL + MOUNT ───────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "requests", "-q"])

from google.colab import drive
drive.mount("/content/drive")

import requests
import csv
import time
import os

# ── CONFIG ────────────────────────────────────────────────────────────
BASE   = "https://api.jolpi.ca/f1/alpha"
FOLDER = "/content/drive/MyDrive/F1 Stats Guy - Jolpica/Jolpica Database"
SLEEP  = 0.5

os.makedirs(FOLDER, exist_ok=True)
print(f"✅ Folder ready: {FOLDER}\n")

# ── FILE PATHS ────────────────────────────────────────────────────────
F_ROUNDS      = f"{FOLDER}/rounds.csv"
F_RACE        = f"{FOLDER}/race_results.csv"
F_QUALI       = f"{FOLDER}/qualifying_results.csv"
F_SPRINT_R    = f"{FOLDER}/sprint_results.csv"
F_SPRINT_Q    = f"{FOLDER}/sprint_qualifying_results.csv"
F_DRIVERS     = f"{FOLDER}/drivers.csv"
F_TEAMS       = f"{FOLDER}/teams.csv"
F_LOG         = f"{FOLDER}/download_log.csv"
F_CHECKPOINT  = f"{FOLDER}/checkpoint.txt"

# ── HEADERS ───────────────────────────────────────────────────────────
H_ROUNDS   = ["round_id","year","round_number","round_name",
               "circuit_id","circuit_name","circuit_reference",
               "country_code","locality","race_date","has_sprint"]

H_RACE     = ["round_id","driver_id","driver_name","team_id",
               "team_name","car_number","position","position_text",
               "is_classified","points","status","grid"]

H_QUALI    = ["round_id","driver_id","driver_name","team_id",
               "team_name","car_number","position",
               "q1_time","q2_time","q3_time"]

H_SPRINT_R = ["round_id","driver_id","driver_name","team_id",
               "team_name","car_number","position","position_text",
               "is_classified","points","status","grid"]

H_SPRINT_Q = ["round_id","driver_id","driver_name","team_id",
               "team_name","car_number","position",
               "sq1_time","sq2_time","sq3_time"]

H_DRIVERS  = ["driver_id","abbreviation","given_name","family_name"]
H_TEAMS    = ["team_id","team_name","primary_color"]
H_LOG      = ["round_id","year","round_number","round_name",
               "r_status","q_status","sr_status","sq_status"]

# ── CHECKPOINT HELPERS ────────────────────────────────────────────────
def read_checkpoint():
    """Returns index of last successfully completed round. -1 if none."""
    if not os.path.exists(F_CHECKPOINT):
        return -1
    with open(F_CHECKPOINT, "r") as f:
        val = f.read().strip()
        return int(val) if val.isdigit() else -1

def write_checkpoint(index):
    """Saves index of last successfully completed round."""
    with open(F_CHECKPOINT, "w") as f:
        f.write(str(index))

# ── CSV HELPERS ────────────────────────────────────────────────────────
def init_csv(filepath, headers):
    """Create CSV with headers if it doesn't exist."""
    if not os.path.exists(filepath):
        with open(filepath, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=headers).writeheader()

def append_csv(filepath, rows, headers):
    """Append rows to existing CSV."""
    if not rows:
        return
    with open(filepath, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=headers).writerows(rows)

# ── INITIALISE ALL FILES ───────────────────────────────────────────────
for path, headers in [
    (F_ROUNDS,   H_ROUNDS),
    (F_RACE,     H_RACE),
    (F_QUALI,    H_QUALI),
    (F_SPRINT_R, H_SPRINT_R),
    (F_SPRINT_Q, H_SPRINT_Q),
    (F_DRIVERS,  H_DRIVERS),
    (F_TEAMS,    H_TEAMS),
    (F_LOG,      H_LOG),
]:
    init_csv(path, headers)

# ── REFERENCE TRACKING ─────────────────────────────────────────────────
# Load already-seen drivers and teams from existing CSVs
# This prevents duplicates in drivers.csv and teams.csv on resume
seen_drivers = set()
seen_teams   = set()

if os.path.exists(F_DRIVERS):
    with open(F_DRIVERS, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            seen_drivers.add(row["driver_id"])

if os.path.exists(F_TEAMS):
    with open(F_TEAMS, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            seen_teams.add(row["team_id"])

print(f"  Existing drivers loaded : {len(seen_drivers)}")
print(f"  Existing teams loaded   : {len(seen_teams)}\n")

# ── API HELPER ─────────────────────────────────────────────────────────
def safe_get(url, retries=3):
    """Fetch URL with retries. Returns JSON or None."""
    for i in range(retries):
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200:
                return r.json()
            elif r.status_code == 404:
                return None
        except Exception as e:
            print(f"  ⚠ Error: {e}")
        time.sleep(2)
    return None

# ── RESULT PARSERS ─────────────────────────────────────────────────────
def track_driver_and_team(driver, team,
                           driver_rows, team_rows):
    """
    Adds driver and team to staging lists if not seen before.
    Staged rows are written only after full round succeeds.
    """
    did = driver.get("id", "")
    tid = team.get("id", "")

    if did and did not in seen_drivers:
        seen_drivers.add(did)
        driver_rows.append({
            "driver_id":    did,
            "abbreviation": driver.get("abbreviation", ""),
            "given_name":   driver.get("given_name", ""),
            "family_name":  driver.get("family_name", ""),
        })

    if tid and tid not in seen_teams:
        seen_teams.add(tid)
        team_rows.append({
            "team_id":       tid,
            "team_name":     team.get("name", ""),
            "primary_color": team.get("primary_color", ""),
        })

def parse_race(round_id, data, driver_rows, team_rows):
    rows = []
    if not data or "data" not in data:
        return rows
    for r in data["data"].get("results", []):
        driver = r.get("driver", {})
        team   = r.get("team", {})
        track_driver_and_team(driver, team, driver_rows, team_rows)
        rows.append({
            "round_id":      round_id,
            "driver_id":     driver.get("id", ""),
            "driver_name":   f"{driver.get('given_name','')} "
                             f"{driver.get('family_name','')}".strip(),
            "team_id":       team.get("id", ""),
            "team_name":     team.get("name", ""),
            "car_number":    r.get("car_number", ""),
            "position":      r.get("position", ""),
            "position_text": r.get("position_text", ""),
            "is_classified": r.get("is_classified", ""),
            "points":        r.get("points", ""),
            "status":        r.get("status", ""),
            "grid":          r.get("grid", ""),
        })
    return rows

def parse_quali(round_id, data, driver_rows, team_rows):
    rows = []
    if not data or "data" not in data:
        return rows
    for r in data["data"].get("results", []):
        driver = r.get("driver", {})
        team   = r.get("team", {})
        comps  = r.get("components", {})
        track_driver_and_team(driver, team, driver_rows, team_rows)
        rows.append({
            "round_id":    round_id,
            "driver_id":   driver.get("id", ""),
            "driver_name": f"{driver.get('given_name','')} "
                           f"{driver.get('family_name','')}".strip(),
            "team_id":     team.get("id", ""),
            "team_name":   team.get("name", ""),
            "car_number":  r.get("car_number", ""),
            "position":    r.get("position", ""),
            "q1_time":     comps.get("Q1", {}).get("time", ""),
            "q2_time":     comps.get("Q2", {}).get("time", ""),
            "q3_time":     comps.get("Q3", {}).get("time", ""),
        })
    return rows

def parse_sprint_quali(round_id, data, driver_rows, team_rows):
    rows = []
    if not data or "data" not in data:
        return rows
    for r in data["data"].get("results", []):
        driver = r.get("driver", {})
        team   = r.get("team", {})
        comps  = r.get("components", {})
        track_driver_and_team(driver, team, driver_rows, team_rows)
        rows.append({
            "round_id":    round_id,
            "driver_id":   driver.get("id", ""),
            "driver_name": f"{driver.get('given_name','')} "
                           f"{driver.get('family_name','')}".strip(),
            "team_id":     team.get("id", ""),
            "team_name":   team.get("name", ""),
            "car_number":  r.get("car_number", ""),
            "position":    r.get("position", ""),
            "sq1_time":    comps.get("SQ1", {}).get("time", ""),
            "sq2_time":    comps.get("SQ2", {}).get("time", ""),
            "sq3_time":    comps.get("SQ3", {}).get("time", ""),
        })
    return rows

# ── PHASE 1: BUILD ROUND INDEX ─────────────────────────────────────────
# Fetches all 1,173 rounds across 12 pages.
# Each round gives us: round_id, year, round_number, name,
# circuit info, race_date, and which sessions exist (R/Q/SR/SQ).
print("═" * 60)
print("PHASE 1 — Building round index")
print("═" * 60)

all_rounds = []
page       = 1

while True:
    data = safe_get(f"{BASE}/core/rounds/?page={page}")
    time.sleep(SLEEP)

    if not data or "data" not in data:
        print(f"  ⚠ No data on page {page} — stopping")
        break

    rounds      = data["data"]
    total_pages = data["metadata"]["total_pages"]
    print(f"  Page {page}/{total_pages} — {len(rounds)} rounds")

    for rnd in rounds:
        session_types = [s["type"] for s in rnd.get("sessions", [])]
        has_sprint    = "SR" in session_types or "SQ" in session_types
        circuit       = rnd.get("circuit", {})

        race_date = ""
        for s in rnd.get("sessions", []):
            if s["type"] == "R":
                race_date = s.get("timestamp", "")[:10]
                break

        all_rounds.append({
            "round_id":          rnd["id"],
            "year":              rnd["season"]["year"],
            "round_number":      rnd["number"],
            "round_name":        rnd["name"],
            "circuit_id":        circuit.get("id", ""),
            "circuit_name":      circuit.get("name", ""),
            "circuit_reference": circuit.get("reference", ""),
            "country_code":      circuit.get("country_code", ""),
            "locality":          circuit.get("locality", ""),
            "race_date":         race_date,
            "has_sprint":        has_sprint,
        })

    if page >= total_pages:
        break
    page += 1

print(f"\n✅ Round index built: {len(all_rounds)} rounds")

# ── PHASE 2: DOWNLOAD RESULTS ──────────────────────────────────────────
# Reads checkpoint to find where to resume from.
# Collects all data for a round in memory first.
# Only writes to CSV after ALL sessions for that round succeed.
# Updates checkpoint after each successful round.
print(f"\n{'═'*60}")
print("PHASE 2 — Downloading results")
print("═" * 60)

checkpoint     = read_checkpoint()
start_index    = checkpoint + 1
skipped        = checkpoint + 1

if checkpoint >= 0:
    print(f"  ▶ Resuming from round {start_index + 1} "
          f"(last completed: {checkpoint + 1})\n")
else:
    print("  ▶ Starting fresh\n")

    # Write rounds.csv on fresh start only
    # On resume, rounds.csv is already complete from previous run
    append_csv(F_ROUNDS, all_rounds, H_ROUNDS)

total_race_rows  = 0
total_quali_rows = 0
total_sr_rows    = 0
total_sq_rows    = 0

for i, rnd in enumerate(all_rounds):

    # Skip already-completed rounds
    if i < start_index:
        continue

    round_id   = rnd["round_id"]
    year       = rnd["year"]
    round_num  = rnd["round_number"]
    round_name = rnd["round_name"]
    has_sprint = rnd["has_sprint"]

    # Progress print every 50 rounds
    if (i - start_index) % 50 == 0:
        print(f"  [{i+1}/{len(all_rounds)}] "
              f"{year} R{round_num} — {round_name}")

    # Staging — collect everything in memory before writing
    race_rows     = []
    quali_rows    = []
    sr_rows       = []
    sq_rows       = []
    driver_rows   = []
    team_rows     = []

    log = {
        "round_id":     round_id,
        "year":         year,
        "round_number": round_num,
        "round_name":   round_name,
        "r_status":     "",
        "q_status":     "",
        "sr_status":    "SKIP",
        "sq_status":    "SKIP",
    }

    # ── Fetch all sessions ────────────────────────────────────────────
    r_data = safe_get(f"{BASE}/results/{round_id}/R/")
    time.sleep(SLEEP)
    if r_data:
        race_rows = parse_race(round_id, r_data,
                               driver_rows, team_rows)
        log["r_status"] = f"OK:{len(race_rows)}"
    else:
        log["r_status"] = "EMPTY"

    q_data = safe_get(f"{BASE}/results/{round_id}/Q/")
    time.sleep(SLEEP)
    if q_data:
        quali_rows = parse_quali(round_id, q_data,
                                 driver_rows, team_rows)
        log["q_status"] = f"OK:{len(quali_rows)}"
    else:
        log["q_status"] = "EMPTY"

    if has_sprint:
        sr_data = safe_get(f"{BASE}/results/{round_id}/SR/")
        time.sleep(SLEEP)
        if sr_data:
            sr_rows = parse_race(round_id, sr_data,
                                 driver_rows, team_rows)
            log["sr_status"] = f"OK:{len(sr_rows)}"
        else:
            log["sr_status"] = "EMPTY"

        sq_data = safe_get(f"{BASE}/results/{round_id}/SQ/")
        time.sleep(SLEEP)
        if sq_data:
            sq_rows = parse_sprint_quali(round_id, sq_data,
                                         driver_rows, team_rows)
            log["sq_status"] = f"OK:{len(sq_rows)}"
        else:
            log["sq_status"] = "EMPTY"

    # ── Commit all data only after full round succeeds ────────────────
    append_csv(F_RACE,     race_rows,   H_RACE)
    append_csv(F_QUALI,    quali_rows,  H_QUALI)
    append_csv(F_SPRINT_R, sr_rows,     H_SPRINT_R)
    append_csv(F_SPRINT_Q, sq_rows,     H_SPRINT_Q)
    append_csv(F_DRIVERS,  driver_rows, H_DRIVERS)
    append_csv(F_TEAMS,    team_rows,   H_TEAMS)
    append_csv(F_LOG,      [log],       H_LOG)

    total_race_rows  += len(race_rows)
    total_quali_rows += len(quali_rows)
    total_sr_rows    += len(sr_rows)
    total_sq_rows    += len(sq_rows)

    # ── Update checkpoint after successful write ───────────────────────
    write_checkpoint(i)

# ── SUMMARY ───────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  DOWNLOAD COMPLETE")
print(f"  Rounds processed   : {len(all_rounds) - start_index}")
print(f"  Race result rows   : {total_race_rows}")
print(f"  Qualifying rows    : {total_quali_rows}")
print(f"  Sprint race rows   : {total_sr_rows}")
print(f"  Sprint quali rows  : {total_sq_rows}")
print(f"  Unique drivers     : {len(seen_drivers)}")
print(f"  Unique teams       : {len(seen_teams)}")
print(f"  Files saved to     : {FOLDER}")
print(f"{'═'*60}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Folder ready: /content/drive/MyDrive/F1 Stats Guy - Jolpica/Jolpica Database

  Existing drivers loaded : 817
  Existing teams loaded   : 203

════════════════════════════════════════════════════════════
PHASE 1 — Building round index
════════════════════════════════════════════════════════════
  Page 1/12 — 100 rounds
  Page 2/12 — 100 rounds
  Page 3/12 — 100 rounds
  Page 4/12 — 100 rounds
  Page 5/12 — 100 rounds
  Page 6/12 — 100 rounds
  Page 7/12 — 100 rounds
  Page 8/12 — 100 rounds
  Page 9/12 — 100 rounds
  Page 10/12 — 100 rounds
  Page 11/12 — 100 rounds
  Page 12/12 — 73 rounds

✅ Round index built: 1173 rounds

════════════════════════════════════════════════════════════
PHASE 2 — Downloading results
════════════════════════════════════════════════════════════
  ▶ Resuming from round 1148 (last completed: 1147)

  [1148/1173] 2025 R23 — Qatar 